# Đọc bảng Gold từ ClickHouse

Notebook này kết nối ClickHouse và đọc 3 bảng Gold:
- `dim_date` — bảng chiều ngày (2020–2030)
- `dim_symbol` — bảng chiều mã chứng khoán HOSE
- `fact_hose_daily_market` — bảng sự kiện OHLCV + chỉ báo kỹ thuật

## 1. Kết nối

In [1]:
import os

import clickhouse_connect
import polars as pl

CLICKHOUSE_HOST = os.getenv("CLICKHOUSE_HOST", "localhost")
CLICKHOUSE_PORT = int(os.getenv("CLICKHOUSE_PORT", "8123"))
CLICKHOUSE_USER = os.getenv("CLICKHOUSE_USER", "admin")
CLICKHOUSE_PASSWORD = os.getenv("CLICKHOUSE_PASSWORD", "admin123")
CLICKHOUSE_DB = os.getenv("CLICKHOUSE_DB", "lakehouse")

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DB,
)

version = client.query("SELECT version()").result_rows[0][0]
print(f"ClickHouse {version} @ {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT} / db={CLICKHOUSE_DB}")

ClickHouse 24.3.18.7 @ localhost:8123 / db=lakehouse


## 2. Danh sách bảng trong database

In [2]:
def query_to_polars(sql: str) -> pl.DataFrame:
    result = client.query(sql)
    return pl.DataFrame(
        {col: [row[i] for row in result.result_rows] for i, col in enumerate(result.column_names)}
    )


tables = query_to_polars(
    f"""
    SELECT
        name,
        engine,
        total_rows,
        formatReadableSize(total_bytes) AS size_on_disk
    FROM system.tables
    WHERE database = '{CLICKHOUSE_DB}'
      AND name NOT LIKE 'smoke%'
    ORDER BY name
    """
)
print(tables)

shape: (16, 4)
┌────────────────────────────┬──────────────────────┬────────────┬──────────────┐
│ name                       ┆ engine               ┆ total_rows ┆ size_on_disk │
│ ---                        ┆ ---                  ┆ ---        ┆ ---          │
│ str                        ┆ str                  ┆ i64        ┆ str          │
╞════════════════════════════╪══════════════════════╪════════════╪══════════════╡
│ alerts_v2                  ┆ MergeTree            ┆ 52         ┆ 4.53 KiB     │
│ dim_date                   ┆ MergeTree            ┆ 4018       ┆ 27.79 KiB    │
│ dim_symbol                 ┆ ReplacingMergeTree   ┆ 5          ┆ 4.04 KiB     │
│ fact_corporate_events      ┆ ReplacingMergeTree   ┆ 236        ┆ 9.83 KiB     │
│ fact_hose_daily_market     ┆ MergeTree            ┆ 605        ┆ 53.05 KiB    │
│ …                          ┆ …                    ┆ …          ┆ …            │
│ realtime_hose_stock_signal ┆ ReplacingMergeTree   ┆ 0          ┆ 0.00 B       │
│

## 3. dim_date

In [4]:
dim_date = query_to_polars("SELECT * FROM dim_date ORDER BY date_key")
print(f"Rows: {len(dim_date):,}  |  Columns: {dim_date.columns}")
dim_date.head(10)

Rows: 4,018  |  Columns: ['date_key', 'full_date', 'day', 'cal_week', 'cal_month', 'cal_quarter', 'cal_year', 'is_weekend', 'event_name', 'event_type', 'is_day_off']


date_key,full_date,day,cal_week,cal_month,cal_quarter,cal_year,is_weekend,event_name,event_type,is_day_off
i64,date,i64,i64,i64,i64,i64,bool,str,str,bool
20200101,2020-01-01,1,1,1,1,2020,false,"""Tết Dương lịch""","""HOLIDAY""",true
20200102,2020-01-02,2,1,1,1,2020,false,null,null,false
20200103,2020-01-03,3,1,1,1,2020,false,null,null,false
20200104,2020-01-04,4,1,1,1,2020,true,null,null,true
20200105,2020-01-05,5,1,1,1,2020,true,null,null,true
20200106,2020-01-06,6,2,1,1,2020,false,null,null,false
20200107,2020-01-07,7,2,1,1,2020,false,null,null,false
20200108,2020-01-08,8,2,1,1,2020,false,null,null,false
20200109,2020-01-09,9,2,1,1,2020,false,null,null,false


In [5]:
# Phân bố ngày nghỉ theo năm
dim_date_stats = query_to_polars(
    """
    SELECT
        cal_year,
        countIf(is_weekend)    AS weekends,
        countIf(is_day_off)    AS day_offs,
        countIf(NOT is_day_off AND NOT is_weekend) AS trading_days
    FROM dim_date
    GROUP BY cal_year
    ORDER BY cal_year
    """
)
print(dim_date_stats)

shape: (11, 4)
┌──────────┬──────────┬──────────┬──────────────┐
│ cal_year ┆ weekends ┆ day_offs ┆ trading_days │
│ ---      ┆ ---      ┆ ---      ┆ ---          │
│ i64      ┆ i64      ┆ i64      ┆ i64          │
╞══════════╪══════════╪══════════╪══════════════╡
│ 2020     ┆ 104      ┆ 114      ┆ 252          │
│ 2021     ┆ 104      ┆ 115      ┆ 250          │
│ 2022     ┆ 105      ┆ 116      ┆ 249          │
│ 2023     ┆ 105      ┆ 116      ┆ 249          │
│ 2024     ┆ 104      ┆ 116      ┆ 250          │
│ …        ┆ …        ┆ …        ┆ …            │
│ 2026     ┆ 104      ┆ 115      ┆ 250          │
│ 2027     ┆ 104      ┆ 115      ┆ 250          │
│ 2028     ┆ 106      ┆ 117      ┆ 249          │
│ 2029     ┆ 104      ┆ 115      ┆ 250          │
│ 2030     ┆ 104      ┆ 115      ┆ 250          │
└──────────┴──────────┴──────────┴──────────────┘


In [10]:
# Các ngày trong năm 2026 có event_type khác NULL và khác 'holiday'
dim_date_stats1 = query_to_polars(
    """
    SELECT
        full_date,
        cal_year,
        cal_month,
        event_type,
        event_name
    FROM dim_date
    WHERE cal_year = 2026 
      AND lowerUTF8(trim(BOTH ' ' FROM event_type)) == 'holiday'
    """
)
print(dim_date_stats1)

shape: (11, 5)
┌────────────┬──────────┬───────────┬────────────┬──────────────────────────┐
│ full_date  ┆ cal_year ┆ cal_month ┆ event_type ┆ event_name               │
│ ---        ┆ ---      ┆ ---       ┆ ---        ┆ ---                      │
│ date       ┆ i64      ┆ i64       ┆ str        ┆ str                      │
╞════════════╪══════════╪═══════════╪════════════╪══════════════════════════╡
│ 2026-01-01 ┆ 2026     ┆ 1         ┆ HOLIDAY    ┆ Tết Dương lịch           │
│ 2026-02-16 ┆ 2026     ┆ 2         ┆ HOLIDAY    ┆ Giao thừa Tết Nguyên Đán │
│ 2026-02-17 ┆ 2026     ┆ 2         ┆ HOLIDAY    ┆ Tết Nguyên Đán           │
│ 2026-02-18 ┆ 2026     ┆ 2         ┆ HOLIDAY    ┆ Mùng hai Tết Nguyên Đán  │
│ 2026-02-19 ┆ 2026     ┆ 2         ┆ HOLIDAY    ┆ Mùng ba Tết Nguyên Đán   │
│ …          ┆ …        ┆ …         ┆ …          ┆ …                        │
│ 2026-04-26 ┆ 2026     ┆ 4         ┆ HOLIDAY    ┆ Ngày Giỗ Tổ Hùng Vương   │
│ 2026-04-30 ┆ 2026     ┆ 4         ┆ HOLIDAY    

## 4. dim_symbol

In [6]:
dim_symbol = query_to_polars("SELECT * FROM dim_symbol ORDER BY symbol")
print(f"Rows: {len(dim_symbol):,}  |  Columns: {dim_symbol.columns}")
dim_symbol.head(10)

Rows: 5  |  Columns: ['symbol_key', 'symbol', 'company_name', 'sector_name', 'company_profile', 'listing_date', 'exchange_code', 'listed_status', 'updated_at']


symbol_key,symbol,company_name,sector_name,company_profile,listing_date,exchange_code,listed_status,updated_at
i64,str,str,str,str,date,str,str,datetime[μs]
1,"""FPT""","""Công ty Cổ phần FPT""","""Technology""","""Công ty Cổ phần FPT (FPT) có t…",2006-12-13,"""HOSE""","""LISTED""",2026-07-02 18:26:55.693759
2,"""HPG""","""Công ty Cổ phần Tập đoàn Hòa P…","""Basic Resources""","""Công ty Cổ phần Tập đoàn Hoà P…",2007-11-15,"""HOSE""","""LISTED""",2026-07-02 18:26:55.693759
3,"""MWG""","""Công ty Cổ phần Đầu tư Thế Giớ…","""Retail""","""Công ty Cổ phần Đầu tư Thế Giớ…",2014-07-14,"""HOSE""","""LISTED""",2026-07-02 18:26:55.693759
4,"""VCB""","""Ngân hàng Thương mại Cổ phần N…","""Banks""","""Ngân hàng Thương mại Cổ phần N…",2009-06-30,"""HOSE""","""LISTED""",2026-07-02 18:26:55.693759
5,"""VNM""","""Công ty Cổ phần Sữa Việt Nam""","""Food & Beverage""","""Công ty Cổ phần Sữa Việt Nam (…",2006-01-19,"""HOSE""","""LISTED""",2026-07-02 18:26:55.693759


In [7]:
# Phân bố theo sector
sector_dist = query_to_polars(
    """
    SELECT
        coalesce(sector_name, '(unknown)') AS sector,
        count() AS symbol_count
    FROM dim_symbol
    GROUP BY sector
    ORDER BY symbol_count DESC
    LIMIT 15
    """
)
print(sector_dist)

shape: (5, 2)
┌─────────────────┬──────────────┐
│ sector          ┆ symbol_count │
│ ---             ┆ ---          │
│ str             ┆ i64          │
╞═════════════════╪══════════════╡
│ Retail          ┆ 1            │
│ Banks           ┆ 1            │
│ Technology      ┆ 1            │
│ Basic Resources ┆ 1            │
│ Food & Beverage ┆ 1            │
└─────────────────┴──────────────┘


## 5. fact_hose_daily_market

In [28]:
# Tổng quan partition theo tháng
fact_monthly = query_to_polars(
    """
    SELECT
        toYYYYMM(trading_date) AS yyyymm,
        count()                AS rows,
        count(DISTINCT symbol_key) AS symbols
    FROM fact_hose_daily_market
    GROUP BY yyyymm
    ORDER BY yyyymm
    """
)
print(f"Partitions: {len(fact_monthly)}")
print(fact_monthly)

Partitions: 6
shape: (6, 3)
┌────────┬──────┬─────────┐
│ yyyymm ┆ rows ┆ symbols │
│ ---    ┆ ---  ┆ ---     │
│ i64    ┆ i64  ┆ i64     │
╞════════╪══════╪═════════╡
│ 202601 ┆ 100  ┆ 5       │
│ 202602 ┆ 75   ┆ 5       │
│ 202603 ┆ 110  ┆ 5       │
│ 202604 ┆ 100  ┆ 5       │
│ 202605 ┆ 100  ┆ 5       │
│ 202606 ┆ 55   ┆ 5       │
└────────┴──────┴─────────┘


In [5]:
# Đọc toàn bộ fact (nếu dữ liệu lớn, giới hạn bằng WHERE trading_date >= '2025-01-01')
fact = query_to_polars(
    """
    SELECT *
    FROM fact_hose_daily_market
    ORDER BY trading_date DESC, symbol_key
    LIMIT 50000
    """
)
print(f"Rows loaded: {len(fact):,}  |  Columns: {fact.columns}")
fact.head(10)

Rows loaded: 555  |  Columns: ['symbol_key', 'date_key', 'trading_date', 'open_price', 'high_price', 'low_price', 'close_price', 'volume', 'price_change', 'pct_change', 'sma20', 'ema20', 'rsi14', 'macd', 'avg_volume_20d', 'updated_at']


symbol_key,date_key,trading_date,open_price,high_price,low_price,close_price,volume,price_change,pct_change,sma20,ema20,rsi14,macd,avg_volume_20d,updated_at
i64,i64,date,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,datetime[μs]
1,20260618,2026-06-18,72.3,72.3,71.5,71.6,13665800,-0.7,-0.009682,73.4435,73.324128,42.792851,-0.264992,9.6640e6,2026-06-19 08:06:20.357963
2,20260618,2026-06-18,24.05,24.15,23.65,23.65,18040500,-0.35,-0.014583,23.86,23.892819,45.545492,-0.130991,1.7845e7,2026-06-19 08:06:20.357963
3,20260618,2026-06-18,78.6,79.0,78.0,78.7,3234500,-0.4,-0.005057,78.22,78.75464,48.295144,-0.683441,4.08983e6,2026-06-19 08:06:20.357963
4,20260618,2026-06-18,62.3,62.4,61.6,61.6,4650700,-0.6,-0.009646,62.255,61.89512,48.039796,0.008305,4.612375e6,2026-06-19 08:06:20.357963
5,20260618,2026-06-18,59.5,59.7,59.1,59.2,2482800,0.2,0.00339,58.905,59.141495,48.665339,-0.275201,2.633255e6,2026-06-19 08:06:20.357963
1,20260617,2026-06-17,73.3,73.4,72.2,72.3,10652200,-0.9,-0.012295,73.6365,73.505615,45.069759,-0.126594,9573019.2,2026-06-19 04:28:23.331462
2,20260617,2026-06-17,24.2,24.3,24.0,24.0,19039000,-0.2,-0.008264,23.8845,23.918379,50.217267,-0.124007,1.8473e7,2026-06-19 04:28:23.331462
3,20260617,2026-06-17,79.3,79.8,78.7,79.1,4068800,-0.3,-0.003778,78.26,78.760392,49.645707,-0.76639,4.25799e6,2026-06-19 04:28:23.331462
4,20260617,2026-06-17,61.8,62.4,61.6,62.2,6825500,0.4,0.006472,62.42,61.926185,53.085979,0.034945,5.385555e6,2026-06-19 04:28:23.331462


In [19]:
# Top 10 mã có khối lượng giao dịch trung bình cao nhất
top_volume = query_to_polars(
    """
    SELECT
        s.symbol,
        round(avg(f.volume), 0)            AS avg_volume,
        round(avg(f.close_price), 2)       AS avg_close,
        round(avg(f.pct_change) * 100, 4)  AS avg_pct_change_pct,
        count()                            AS trading_days
    FROM fact_hose_daily_market f
    JOIN dim_symbol s ON s.symbol_key = f.symbol_key
    GROUP BY s.symbol
    ORDER BY avg_volume DESC
    LIMIT 10
    """
)
print("Top 10 mã theo khối lượng trung bình:")
print(top_volume)

Top 10 mã theo khối lượng trung bình:
shape: (5, 5)
┌────────┬─────────────┬───────────┬────────────────────┬──────────────┐
│ symbol ┆ avg_volume  ┆ avg_close ┆ avg_pct_change_pct ┆ trading_days │
│ ---    ┆ ---         ┆ ---       ┆ ---                ┆ ---          │
│ str    ┆ f64         ┆ f64       ┆ f64                ┆ i64          │
╞════════╪═════════════╪═══════════╪════════════════════╪══════════════╡
│ HPG    ┆ 3.8015123e7 ┆ 24.23     ┆ 0.0637             ┆ 108          │
│ FPT    ┆ 1.1561328e7 ┆ 82.29     ┆ -0.2002            ┆ 108          │
│ VCB    ┆ 9.820305e6  ┆ 62.97     ┆ 0.096              ┆ 108          │
│ MWG    ┆ 7.096827e6  ┆ 83.86     ┆ -0.0615            ┆ 108          │
│ VNM    ┆ 5.826033e6  ┆ 63.21     ┆ 0.0122             ┆ 108          │
└────────┴─────────────┴───────────┴────────────────────┴──────────────┘


In [30]:
# Lấy lịch sử một mã cụ thể (thay 'VCB' bằng mã bạn muốn xem)
SYMBOL = "VCB"

symbol_history = query_to_polars(
    f"""
    SELECT
        f.trading_date,
        f.open_price,
        f.high_price,
        f.low_price,
        f.close_price,
        f.volume,
        f.pct_change,
        f.sma20,
        f.ema20,
        f.rsi14,
        f.macd
    FROM fact_hose_daily_market f
    JOIN dim_symbol s ON s.symbol_key = f.symbol_key
    WHERE s.symbol = '{SYMBOL}'
    ORDER BY f.trading_date
    """
)
print(f"{SYMBOL}: {len(symbol_history)} ngày giao dịch")
symbol_history.head(20)

VCB: 108 ngày giao dịch


trading_date,open_price,high_price,low_price,close_price,volume,pct_change,sma20,ema20,rsi14,macd
date,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64
2026-01-05,57.5,57.6,57.0,57.1,3591800,null,null,null,null,null
2026-01-06,57.1,57.5,56.6,57.3,4173200,0.003503,null,null,null,null
2026-01-07,57.6,59.9,57.5,59.6,12918300,0.04014,null,null,null,null
2026-01-08,59.8,63.7,59.7,63.7,27104600,0.068792,null,null,null,null
2026-01-09,65.4,68.0,65.4,68.0,32481400,0.067504,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…
2026-01-26,68.9,71.1,68.7,69.6,10990600,0.014577,null,null,70.170244,null
2026-01-27,69.7,71.5,68.1,70.6,9645200,0.014368,null,null,71.240096,null
2026-01-28,71.1,72.9,69.4,69.6,12283200,-0.014164,null,null,68.590836,null


## 6. Join dim_date + dim_symbol + fact

In [21]:
# Dữ liệu giao dịch tuần gần nhất (full join 3 bảng)
latest_week = query_to_polars(
    """
    SELECT
        d.full_date,
        d.cal_year,
        d.cal_month,
        s.symbol,
        s.sector_name,
        f.open_price,
        f.close_price,
        f.volume,
        f.pct_change,
        f.rsi14
    FROM fact_hose_daily_market f
    JOIN dim_date   d ON d.date_key   = f.date_key
    JOIN dim_symbol s ON s.symbol_key = f.symbol_key
    WHERE f.trading_date >= (SELECT max(trading_date) - 6 FROM fact_hose_daily_market)
    ORDER BY f.trading_date DESC, s.symbol
    """
)
print(f"Rows: {len(latest_week):,}")
latest_week.head(20)

Rows: 25


full_date,cal_year,cal_month,symbol,sector_name,open_price,close_price,volume,pct_change,rsi14
date,i64,i64,str,str,f64,f64,i64,f64,f64
2026-06-15,2026,6,"""FPT""","""Technology""",74.4,73.6,6422700,0.001361,49.513033
2026-06-15,2026,6,"""HPG""","""Basic Resources""",23.55,24.35,30695200,0.049569,55.325553
2026-06-15,2026,6,"""MWG""","""Retail""",77.0,79.4,7871700,0.039267,50.631785
2026-06-15,2026,6,"""VCB""","""Banks""",62.3,61.6,5222300,0.0,48.148905
2026-06-15,2026,6,"""VNM""","""Food & Beverage""",59.3,59.7,2021700,0.011864,53.086572
…,…,…,…,…,…,…,…,…,…
2026-06-10,2026,6,"""FPT""","""Technology""",73.5,74.2,4736000,0.006784,51.223935
2026-06-10,2026,6,"""HPG""","""Basic Resources""",23.55,23.6,7505500,0.002123,41.835492
2026-06-10,2026,6,"""MWG""","""Retail""",77.1,78.2,2866200,0.010336,45.385597


## 7. QA checks — toàn vẹn dữ liệu Gold

Một số kiểm tra nhanh sau backfill (`query_to_polars`, `client`, `pl` đã định nghĩa ở mục 1–2):

1. **`is_day_off = true` mà vẫn có data** — kỳ vọng rỗng (ngày lễ/nghỉ không được có giao dịch).
2. **Ngày giao dịch bị thiếu data (gap)** trong khoảng đã load — kỳ vọng rỗng.
3. **Mỗi mã cùng số phiên** — kỳ vọng distinct = 1.
4. **Vi phạm quy tắc OHLCV** — kỳ vọng rỗng.
5. **Warmup chỉ báo** khớp quy ước (`ema20`/`macd` luôn có; `sma20`/`avg_volume_20d` null 19 phiên đầu; `rsi14` null 14 phiên đầu).

In [22]:
def show_check(title: str, df: pl.DataFrame, ok_when_empty: bool = True) -> pl.DataFrame:
    """In PASS/FAIL. Check 'kỳ vọng rỗng' -> PASS khi df rỗng; ngược lại PASS khi có dòng."""
    n = len(df)
    ok = (n == 0) if ok_when_empty else (n > 0)
    print(f"[{'✅ PASS' if ok else '❌ FAIL'}] {title} — {n} dòng")
    return df


# Check 1: Có ngày is_day_off = true mà VẪN có data trong fact không?  (kỳ vọng: KHÔNG)
dayoff_with_data = query_to_polars(
    """
    SELECT d.full_date, d.event_name, count() AS fact_rows
    FROM fact_hose_daily_market f
    JOIN dim_date d ON d.date_key = f.date_key
    WHERE d.is_day_off = 1
    GROUP BY d.full_date, d.event_name
    ORDER BY d.full_date
    """
)
show_check("Ngày nghỉ (is_day_off=true) lại có data giao dịch", dayoff_with_data, ok_when_empty=True)
dayoff_with_data

[✅ PASS] Ngày nghỉ (is_day_off=true) lại có data giao dịch — 0 dòng


shape: (0, 0)
┌┐
╞╡
└┘

In [23]:
# Check 2: Ngày giao dịch (không cuối tuần & không nghỉ lễ) NẰM TRONG khoảng đã load mà THIẾU data
missing_trading_days = query_to_polars(
    """
    SELECT d.full_date, toDayOfWeek(d.full_date) AS dow, d.event_name
    FROM dim_date d
    WHERE d.is_weekend = 0
      AND d.is_day_off = 0
      AND d.full_date BETWEEN (SELECT min(trading_date) FROM fact_hose_daily_market)
                          AND (SELECT max(trading_date) FROM fact_hose_daily_market)
      AND d.full_date NOT IN (SELECT DISTINCT trading_date FROM fact_hose_daily_market)
    ORDER BY d.full_date
    """
)
show_check("Ngày giao dịch bị thiếu data (gap) trong khoảng đã load", missing_trading_days, ok_when_empty=True)
missing_trading_days

[✅ PASS] Ngày giao dịch bị thiếu data (gap) trong khoảng đã load — 0 dòng


shape: (0, 0)
┌┐
╞╡
└┘

In [24]:
# Check 3: Mỗi mã có cùng số phiên & cùng khoảng ngày không?
per_symbol = query_to_polars(
    """
    SELECT s.symbol,
           count()             AS trading_days,
           min(f.trading_date) AS first_day,
           max(f.trading_date) AS last_day
    FROM fact_hose_daily_market f
    JOIN dim_symbol s ON s.symbol_key = f.symbol_key
    GROUP BY s.symbol
    ORDER BY s.symbol
    """
)
n_distinct = per_symbol["trading_days"].n_unique()
print(f"[{'✅ PASS' if n_distinct == 1 else '❌ FAIL'}] Tất cả mã cùng số phiên — "
      f"distinct = {n_distinct} (kỳ vọng 1)")
per_symbol

[✅ PASS] Tất cả mã cùng số phiên — distinct = 1 (kỳ vọng 1)


symbol,trading_days,first_day,last_day
str,i64,date,date
"""FPT""",108,2026-01-05,2026-06-15
"""HPG""",108,2026-01-05,2026-06-15
"""MWG""",108,2026-01-05,2026-06-15
"""VCB""",108,2026-01-05,2026-06-15
"""VNM""",108,2026-01-05,2026-06-15


In [25]:
# Check 4: Vi phạm quy tắc OHLCV (high>=low, high>=open/close, low<=open/close, giá>0, volume>=0)
ohlcv_violations = query_to_polars(
    """
    SELECT trading_date, symbol_key, open_price, high_price, low_price, close_price, volume
    FROM fact_hose_daily_market
    WHERE NOT (
        high_price >= low_price
        AND high_price >= open_price AND high_price >= close_price
        AND low_price  <= open_price AND low_price  <= close_price
        AND low_price > 0 AND volume >= 0
    )
    ORDER BY trading_date, symbol_key
    """
)
show_check("Vi phạm quy tắc OHLCV", ohlcv_violations, ok_when_empty=True)
ohlcv_violations

[✅ PASS] Vi phạm quy tắc OHLCV — 0 dòng


shape: (0, 0)
┌┐
╞╡
└┘

In [31]:
# Check 5: Số null của chỉ báo có khớp quy ước warmup không? (mọi chỉ báo đều null-warmup)
#   sma20, avg_volume_20d, ema20 -> null 19 phiên đầu mỗi mã (đủ 20 phiên mới có)
#   rsi14                        -> null 14 phiên đầu mỗi mã
#   macd  (= ema12 - ema26)      -> null 25 phiên đầu mỗi mã (đủ 26 phiên, leg ema26)
n_symbols = query_to_polars("SELECT count() AS n FROM dim_symbol")["n"][0]
warmup = query_to_polars(
    """
    SELECT
        countIf(sma20 IS NULL)          AS sma20_null,
        countIf(avg_volume_20d IS NULL) AS avgvol_null,
        countIf(ema20 IS NULL)          AS ema20_null,
        countIf(rsi14 IS NULL)          AS rsi14_null,
        countIf(macd IS NULL)           AS macd_null
    FROM fact_hose_daily_market
    """
)
exp_w20, exp_rsi, exp_macd = n_symbols * 19, n_symbols * 14, n_symbols * 25
r = warmup.row(0, named=True)
ok = (
    r["sma20_null"] == exp_w20 and r["avgvol_null"] == exp_w20 and r["ema20_null"] == exp_w20
    and r["rsi14_null"] == exp_rsi and r["macd_null"] == exp_macd
)
print(f"[{'✅ PASS' if ok else '❌ FAIL'}] Warmup chỉ báo — kỳ vọng: "
      f"sma20=avgvol=ema20={exp_w20}, rsi14={exp_rsi}, macd={exp_macd} (n_symbols={n_symbols})")
warmup

[✅ PASS] Warmup chỉ báo — kỳ vọng: sma20=avgvol=ema20=95, rsi14=70, macd=125 (n_symbols=5)


sma20_null,avgvol_null,ema20_null,rsi14_null,macd_null
i64,i64,i64,i64,i64
95,95,95,70,125


## 8. fact_hose_index_daily

Bảng fact chỉ số thị trường (VN-Index, VN30…). Cùng shape OHLCV + chỉ báo như `fact_hose_daily_market`, nhưng định danh bằng **natural key `index_code`** (không có `symbol_key`, không join `dim_symbol`).

In [7]:
# Tổng quan partition theo tháng + số chỉ số
index_monthly = query_to_polars(
    """
    SELECT
        toYYYYMM(trading_date)      AS yyyymm,
        count()                     AS rows,
        count(DISTINCT index_code)  AS indices
    FROM fact_hose_index_daily
    GROUP BY yyyymm
    ORDER BY yyyymm
    """
)
print(f"Partitions: {len(index_monthly)}")
print(index_monthly)

# Đọc toàn bộ fact index (định danh bằng natural key index_code, KHÔNG join dim_symbol)
fact_index = query_to_polars(
    """
    SELECT *
    FROM fact_hose_index_daily
    WHERE index_code = 'VNINDEX'
    ORDER BY trading_date DESC 
    LIMIT 50000

    """
)
print(f"Rows loaded: {len(fact_index):,}  |  Columns: {fact_index.columns}")
fact_index.tail(60)

Partitions: 3
shape: (3, 3)
┌────────┬──────┬─────────┐
│ yyyymm ┆ rows ┆ indices │
│ ---    ┆ ---  ┆ ---     │
│ i64    ┆ i64  ┆ i64     │
╞════════╪══════╪═════════╡
│ 202604 ┆ 14   ┆ 2       │
│ 202605 ┆ 40   ┆ 2       │
│ 202606 ┆ 30   ┆ 2       │
└────────┴──────┴─────────┘
Rows loaded: 42  |  Columns: ['index_code', 'date_key', 'trading_date', 'open_price', 'high_price', 'low_price', 'close_price', 'volume', 'price_change', 'pct_change', 'sma20', 'ema20', 'rsi14', 'macd', 'avg_volume_20d', 'updated_at']


index_code,date_key,trading_date,open_price,high_price,low_price,close_price,volume,price_change,pct_change,sma20,ema20,rsi14,macd,avg_volume_20d,updated_at
str,i64,date,f64,f64,f64,f64,i64,f64,f64,f64,f64,f64,f64,f64,datetime[μs]
"""VNINDEX""",20260619,2026-06-19,1837.38,1838.52,1825.51,1828.95,233395930,-1.52,-0.00083,1829.1345,1828.630774,49.035432,-15.406327,6.3847e8,2026-06-19 06:13:22.159643
"""VNINDEX""",20260618,2026-06-18,1822.37,1836.4,1819.62,1830.47,633590659,24.27,0.013437,1831.5435,1828.597171,49.51938,-17.653531,6.6951e8,2026-06-19 06:13:22.159643
"""VNINDEX""",20260617,2026-06-17,1797.41,1808.57,1787.89,1806.2,821620763,-1.74,-0.000962,1834.8645,1828.400031,40.866433,-20.419766,6.7169e8,2026-06-19 06:13:22.159643
"""VNINDEX""",20260616,2026-06-16,1808.56,1811.59,1799.86,1807.94,672837809,8.63,0.004796,1840.216,1830.736876,41.338154,-21.182346,6.9049e8,2026-06-19 06:13:22.159643
"""VNINDEX""",20260615,2026-06-15,1804.24,1810.41,1775.72,1799.31,792064448,7.66,0.004275,1845.4655,1833.136548,38.044523,-22.024732,7.07231517e8,2026-06-19 06:13:22.159643
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""VNINDEX""",20260424,2026-04-24,1873.23,1881.93,1843.63,1853.29,673852010,-17.07,-0.009127,null,null,null,null,null,2026-06-19 06:13:22.159643
"""VNINDEX""",20260423,2026-04-23,1868.31,1888.99,1855.09,1870.36,1030969130,13.06,0.007032,null,null,null,null,null,2026-06-19 06:13:22.159643
"""VNINDEX""",20260422,2026-04-22,1834.53,1861.34,1819.21,1857.3,708137323,23.82,0.012992,null,null,null,null,null,2026-06-19 06:13:22.159643


## 9. fact_corporate_events

Bảng factless fact sự kiện doanh nghiệp (cổ tức, phát hành, GD nội bộ, ĐHĐCĐ, niêm yết thêm…). KHÔNG indicator/rolling, chỉ FK tới `dim_symbol` (`symbol_key`) + `dim_date` (`date_key`), giữ `symbol` denormalized cho serving. Unpartitioned, overwrite full theo natural key `event_id`.

In [8]:
# Tổng quan: số sự kiện theo loại (event_code / event_label) và theo mã
events_by_code = query_to_polars(
    """
    SELECT
        event_code,
        any(event_label)        AS event_label,
        count()                 AS events,
        count(DISTINCT symbol)  AS symbols,
        min(event_date)         AS first_event,
        max(event_date)         AS last_event
    FROM fact_corporate_events
    GROUP BY event_code
    ORDER BY events DESC
    """
)
print(f"Số loại sự kiện: {len(events_by_code)}")
print(events_by_code)

events_by_symbol = query_to_polars(
    """
    SELECT symbol, count() AS events, count(DISTINCT event_code) AS event_types
    FROM fact_corporate_events
    GROUP BY symbol
    ORDER BY events DESC
    """
)
print("\nSố sự kiện theo mã:")
print(events_by_symbol)

# Đọc toàn bộ bảng (nhỏ, unpartitioned), mới nhất trước
events = query_to_polars(
    """
    SELECT *
    FROM fact_corporate_events
    ORDER BY event_date DESC, symbol, event_id
    LIMIT 50000
    """
)
print(f"\nRows loaded: {len(events):,}  |  Columns: {events.columns}")
events.head(30)

Số loại sự kiện: 9
shape: (9, 6)
┌────────────┬───────────────────────┬────────┬─────────┬─────────────┬────────────┐
│ event_code ┆ event_label           ┆ events ┆ symbols ┆ first_event ┆ last_event │
│ ---        ┆ ---                   ┆ ---    ┆ ---     ┆ ---         ┆ ---        │
│ str        ┆ str                   ┆ i64    ┆ i64     ┆ date        ┆ date       │
╞════════════╪═══════════════════════╪════════╪═════════╪═════════════╪════════════╡
│ DDINS      ┆ GD nội bộ (tổ chức)   ┆ 89     ┆ 5       ┆ 2021-09-30  ┆ 2026-06-03 │
│ DDIND      ┆ GD nội bộ (cá nhân)   ┆ 53     ┆ 4       ┆ 2020-04-22  ┆ 2026-07-07 │
│ ISS        ┆ Phát hành cổ phiếu    ┆ 23     ┆ 4       ┆ 2020-06-26  ┆ 2026-06-30 │
│ DDRP       ┆ GD nội bộ (liên quan) ┆ 18     ┆ 4       ┆ 2020-12-28  ┆ 2026-07-07 │
│ AGME       ┆ ĐHĐCĐ thường niên     ┆ 17     ┆ 5       ┆ 2020-06-26  ┆ 2026-04-24 │
│ DIV        ┆ Cổ tức tiền mặt       ┆ 16     ┆ 5       ┆ 2020-12-21  ┆ 2026-06-26 │
│ AIS        ┆ Niêm yết thêm    

event_id,symbol_key,date_key,symbol,event_date,event_code,event_label,title_vi,value,updated_at
str,i64,i64,str,date,str,str,str,f64,datetime[μs]
"""6a445e54279ac17a86e4cac0""",3,20260707,"""MWG""",2026-07-07,"""DDIND""","""GD nội bộ (cá nhân)""","""Vũ Đăng Linh - Đăng kí Mua 275…",null,2026-07-02 18:29:17.587645
"""6a445e54279ac17a86e4cac1""",3,20260707,"""MWG""",2026-07-07,"""DDIND""","""GD nội bộ (cá nhân)""","""Lý Trần Kim Ngân - Đăng kí Mua…",null,2026-07-02 18:29:17.587645
"""6a445e54279ac17a86e4cac2""",3,20260707,"""MWG""",2026-07-07,"""DDIND""","""GD nội bộ (cá nhân)""","""Lê Thị Thu Trang - Đăng kí Mua…",null,2026-07-02 18:29:17.587645
"""6a445e54279ac17a86e4cac3""",3,20260707,"""MWG""",2026-07-07,"""DDIND""","""GD nội bộ (cá nhân)""","""Đoàn Văn Hiểu Em - Đăng kí Mua…",null,2026-07-02 18:29:17.587645
"""6a445e54279ac17a86e4cac4""",3,20260707,"""MWG""",2026-07-07,"""DDRP""","""GD nội bộ (liên quan)""","""Trịnh Quang Khải - Đăng kí Mua…",null,2026-07-02 18:29:17.587645
…,…,…,…,…,…,…,…,…,…
"""6a1a2e226c972745c9ee889d""",5,20260603,"""VNM""",2026-06-03,"""DDINS""","""GD nội bộ (tổ chức)""","""Công ty TNHH MTV Đầu Tư SCIC -…",null,2026-07-02 18:29:17.587645
"""6a0fa234a6ca51ba73a2a0a0""",1,20260528,"""FPT""",2026-05-28,"""DIV""","""Cổ tức tiền mặt""","""Trả cổ tức bằng tiền mặt - Đợt…",1000.0,2026-07-02 18:29:17.587645
"""69eab80f9a7ad236002b1866""",2,20260525,"""HPG""",2026-05-25,"""ISS""","""Phát hành cổ phiếu""","""Phát hành cổ phiếu - Trả Cổ tứ…",null,2026-07-02 18:29:17.587645
